# Deliverable 5 — Evaluate trained models

Loads the best checkpoints created by Deliverable 4 and reports Dice, IoU, precision, recall, and pixel accuracy using the project's metric implementation. By default this evaluates the validation split. Change `EVALUATION_SPLIT` to `"test"` only if a labeled test split is available. Results are saved in this submission folder.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import torch
from torch.utils.data import DataLoader

def locate(name):
    for candidate in (Path.cwd(), Path.cwd().parent / name, Path.cwd().parent.parent / name):
        if candidate.name == name and candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not locate sibling folder: {name}")

SUBMISSION_DIR = locate("oil_spill_detection_submission")
candidates = (Path.cwd(), Path.cwd().parent / "oil_spill_detection", Path.cwd().parent.parent / "oil_spill_detection")
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "src" / "evaluate.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project folder. Open the notebook from the project or submission folder.")
sys.path.insert(0, str(PROJECT_ROOT))
from config import IMAGE_SIZE
from src.data import SARSegmentationDataset
from src.metrics import segmentation_metrics
from src.models import SmallUNet, UNet

EVALUATION_SPLIT = "val"  # Use "test" only when a labeled test split exists.
BATCH_SIZE = 4
CHECKPOINT_DIR = SUBMISSION_DIR / "deliverable_4_outputs" / "models"
OUTPUT_DIR = SUBMISSION_DIR / "deliverable_5_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT = PROJECT_ROOT / "data" / "raw"  # Change this if your dataset is elsewhere.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Evaluation split:", EVALUATION_SPLIT)
print("Device:", device)

In [ ]:
dataset = SARSegmentationDataset(DATA_ROOT, EVALUATION_SPLIT, IMAGE_SIZE, augment=False)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
if len(dataset) == 0:
    raise RuntimeError(f"No examples found in the {EVALUATION_SPLIT} split.")

model_specs = (("SmallUNet", SmallUNet), ("UNet", UNet))
results = {}
for name, model_class in model_specs:
    checkpoint_path = CHECKPOINT_DIR / f"{name.lower()}_best.pt"
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Missing {checkpoint_path}. Run Deliverable 4 first.")
    model = model_class().to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    probabilities, targets = [], []
    with torch.no_grad():
        for images, masks in loader:
            logits = model(images.to(device))
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
            targets.append(masks.numpy())
    results[name] = segmentation_metrics(np.concatenate(probabilities), np.concatenate(targets))

results_path = OUTPUT_DIR / f"{EVALUATION_SPLIT}_metrics.json"
results_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
display(results)
print("Metrics saved:", results_path)
print("Compare Dice/IoU and precision/recall; pixel accuracy alone can be misleading for sparse oil masks.")